In [2]:
!pip install wandb gensim sentence-transformers transformers sentencepiece faiss-cpu -q

import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import os
os.environ["WANDB_SILENT"] = "true"

from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
w_key = user_secrets.get_secret("WANDB_Key")
wandb.login(key=w_key)

import torch
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration, pipeline,

OPTS   = ['A', 'B', 'C', 'D', 'E']
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| device:", device)
train.head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.2 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [ ]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

In [ ]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util as st_util
import faiss

hf_train = Dataset.from_pandas(train)
hf_test  = Dataset.from_pandas(test)
print(hf_train)

# tfidf on full corpus so test vocab is covered
full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=25000, sublinear_tf=True)
tfidf.fit(full_corpus)
print(f"tfidf vocab: {len(tfidf.vocabulary_)}")

# word2vec on cleaned stopword-removed tokens
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))
w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
print(f"w2v vocab: {len(w2v_model.wv)}")

# roberta for context aware embeddings and attention inspection
bert_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
bert_model     = AutoModel.from_pretrained("roberta-base", output_attentions=True).to(device)
bert_model.eval()
sample_inputs = bert_tokenizer(train['prompt'].iloc[0], return_tensors="pt").to(device)
with torch.no_grad():
    sample_out = bert_model(**sample_inputs)
print(f"\nattention layers: {len(sample_out.attentions)}, heads per layer: {sample_out.attentions[0].shape[1]}")
print("this is how roberta learns which words to focus on relative to others")

# zero shot nli pipeline deberta is stronger than bart for entailment tasks
zero_shot = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-large",
    device=0 if device == "cuda" else -1
)
print("zero-shot pipeline loaded: nli-deberta-v3-large")

# minilm for building the retrieval vector database
# fast and good enough for dense retrieval
minilm  = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# knowledge base: prompt + correct answer so retrieved docs give useful signal
kb_docs = [f"{row['prompt']} Answer: {row[row['answer']]}" for _, row in train.iterrows()]
kb_embs = minilm.encode(kb_docs, batch_size=32, show_progress_bar=True).astype(np.float32)
faiss.normalize_L2(kb_embs)

# IndexFlatIP = exact inner product search, equivalent to cosine sim on normalized vecs
faiss_index = faiss.IndexFlatIP(kb_embs.shape[1])
faiss_index.add(kb_embs)
print(f"faiss index built: {faiss_index.ntotal} documents indexed")

In [ ]:
def tfidf_cosine(text1, text2):
    v1 = tfidf.transform([text1])
    v2 = tfidf.transform([text2])
    return float(cosine_similarity(v1, v2)[0][0])

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1)
    v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

# roberta mean pooling ignores padding tokens via attention mask
def get_roberta_embedding(text):
    inputs     = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
    with torch.no_grad():
        out    = bert_model(**inputs)
    token_embs = out.last_hidden_state
    mask       = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
    summed     = torch.sum(token_embs * mask, dim=1)
    counts     = torch.clamp(mask.sum(dim=1), min=1e-9)
    return (summed / counts).squeeze(0).cpu().numpy()

def rank_tfidf(row):
    sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_roberta(row):
    pv   = get_roberta_embedding(row['prompt']).reshape(1, -1)
    sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

# shared helper returns raw scores dict so both zeroshot and rag can use it
# framing as hypothesis improves NLI model accuracy over raw option text
def zeroshot_scores(row, prompt_override=None):
    prompt           = prompt_override if prompt_override else row['prompt']
    candidate_labels = [f"The correct answer is: {row[o]}" for o in OPTS]
    label_to_letter  = {f"The correct answer is: {row[o]}": o for o in OPTS}
    result           = zero_shot(prompt, candidate_labels, multi_label=True)
    return {label_to_letter[l]: s for l, s in zip(result['labels'], result['scores'])}

def rank_zeroshot(row):
    scores = zeroshot_scores(row)
    return sorted(scores, key=lambda x: scores[x], reverse=True)

# retrieve similar training examples from faiss and prepend as context
# exclude_self=True when evaluating on train so the question itslf isnt retrieved
def retrieve_context(query, k=3, exclude_self=False):
    q_emb = minilm.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q_emb)
    n = k + 1 if exclude_self else k
    _, indices = faiss_index.search(q_emb, n)
    idxs = indices[0][1:] if exclude_self else indices[0]
    return [kb_docs[i] for i in idxs]

def rank_rag(row, exclude_self=False):
    contexts  = retrieve_context(row['prompt'], k=3, exclude_self=exclude_self)
    augmented = " | ".join(contexts) + " Now answer this: " + row['prompt']
    scores    = zeroshot_scores(row, prompt_override=augmented)
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

actuals        = train['answer'].tolist()
top3_maj       = list(train['answer'].value_counts().index[:3])
map3_maj       = round(mapk(actuals, [top3_maj] * len(train)), 4)
tfidf_preds    = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
map3_tfidf     = round(mapk(actuals, tfidf_preds), 4)
w2v_preds      = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v       = round(mapk(actuals, w2v_preds), 4)
sample_roberta = train.sample(n=200, random_state=42)
roberta_preds  = [rank_roberta(row)[:3] for _, row in sample_roberta.iterrows()]
map3_roberta   = round(mapk(sample_roberta['answer'].tolist(), roberta_preds), 4)

print("running zero-shot on full train...")
zs_preds  = [rank_zeroshot(row)[:3] for _, row in train.iterrows()]
map3_zs   = round(mapk(actuals, zs_preds), 4)

print("running RAG on full train...")
rag_preds = [rank_rag(row, exclude_self=True)[:3] for _, row in train.iterrows()]
map3_rag  = round(mapk(actuals, rag_preds), 4)

print(f"\nmajority baseline    MAP@3 : {map3_maj}")
print(f"tfidf cosine         MAP@3 : {map3_tfidf}")
print(f"word2vec cosine      MAP@3 : {map3_w2v}")
print(f"roberta cosine (200) MAP@3 : {map3_roberta}")
print(f"zero-shot nli        MAP@3 : {map3_zs}")
print(f"rag + zero-shot      MAP@3 : {map3_rag}")

run = wandb.init(
    project = "23f1000362-t22026",
    name    = "milestone3-rag-pipeline",
    config  = {
        "milestone": 3, "retriever": "all-MiniLM-L6-v2",
        "vector_db": "faiss-flatIP", "kb_size": len(kb_docs),
        "ranker": "cross-encoder/nli-deberta-v3-large", "top_k_retrieval": 3
    },
    tags = ["milestone-3", "rag"]
)
wandb.log({
    "map3/majority_baseline"  : map3_maj,
    "map3/tfidf_cosine"       : map3_tfidf,
    "map3/word2vec_cosine"    : map3_w2v,
    "map3/roberta_cosine_200" : map3_roberta,
    "map3/zeroshot_nli"       : map3_zs,
    "map3/rag_zeroshot"       : map3_rag
})
wandb.finish()

best_model = "rag" if map3_rag > map3_zs else "zeroshot"
print(f"\nbest model on train: {best_model}")

test_preds = []
for _, row in test.iterrows():
    ranked = rank_rag(row, exclude_self=False) if best_model == "rag" else rank_zeroshot(row)
    test_preds.append(" ".join(ranked[:3]))

submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
submission.head(5)

In [15]:
#Milestone 3
import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline, BertTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 
from sentence_transformers import CrossEncoder
import wandb
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

model    = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index    = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

zs            = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0 if device=="cuda" else -1)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

row_150    = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150[o]) for o in ['A','B','C','D','E']]
ans_150    = str(row_150[row_150['answer']])

print(f"kb size: {len(kb)}")
print(f"iitm index: {index.ntotal} docs")
print(f"row 150 correct answer: {row_150['answer']} = {ans_150[:60]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

kb size: 2000
iitm index: 2000 docs
row 150 correct answer: C = The butterfly effect is the phenomenon that a small change i


In [17]:
run = wandb.init(
    project="23f1000362-t22026",
    name="milestone3-graded-questions",
    config={"milestone": 3, "questions": 8},
    tags=["milestone-3", "graded"]
)

# Q1 zero shot on row 150, score of ground truth option
res_q1          = zs(prompt_150, labels_150)
score_map_q1    = dict(zip(res_q1['labels'], res_q1['scores']))
q1              = round(score_map_q1[ans_150], 3)
print(f" Q1: {q1}")

# Q2 embed prompt 150, retrieve top 10, find rank of true doc (index 150 in kb)
q_emb_150       = model.encode([prompt_150])
D2, I2          = index.search(q_emb_150, 10)
retrieved_indices = I2[0].tolist()
q2              = retrieved_indices.index(150) + 1 if 150 in retrieved_indices else "not in top 10"
print(f"retrieved order: {retrieved_indices}")
print(f" Q2: {q2}")

# Q3 cross encoder reranking of same top 10, find rank of true doc
cross_encoder   = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10         = [kb[i] for i in retrieved_indices]
pairs           = [[prompt_150, doc] for doc in docs_10]
ce_scores       = cross_encoder.predict(pairs)
ranked_idxs     = [retrieved_indices[i] for i in np.argsort(ce_scores)[::-1]]
q3              = ranked_idxs.index(150) + 1 if 150 in ranked_idxs else "not found"
print(f"cross encoder ranked order: {ranked_idxs}")
print(f" Q3: {q3}")

# Q4 top 5 for row 42, concat, tokenize with bert-base-uncased without truncation
bert_tok_q4     = BertTokenizer.from_pretrained("bert-base-uncased")
row_42          = train.iloc[42]
prompt_42       = str(row_42['prompt'])
_, I42          = index.search(model.encode([prompt_42]), 5)
docs_42         = [kb[i] for i in I42[0]]
rag_str_42      = f"Context: {' '.join(docs_42)} Question: {prompt_42}"
q4              = len(bert_tok_q4(rag_str_42)['input_ids'])
print(f" Q4: {q4}")

# Q5 true doc for row 150 as context, run zero shot, score of correct option
true_doc_150    = kb[150]
rag_str_150     = f"Context: {true_doc_150} Question: {prompt_150}"
res_q5          = zs(rag_str_150, labels_150)
score_map_q5    = dict(zip(res_q5['labels'], res_q5['scores']))
q5              = round(score_map_q5[ans_150], 3)
print(f" Q5: {q5}")

# Q6 adversarial RAG force wrong context from kb index 999
adv_rag_str     = f"Context: {kb[999]} Question: {prompt_150}"
res_q6          = zs(adv_rag_str, labels_150)
score_map_q6    = dict(zip(res_q6['labels'], res_q6['scores']))
q6              = round(score_map_q6[ans_150], 3)
print(f" Q6: {q6}")

# Q7  hit rate for first 100 rows, k=5
hits = 0
for i in range(100):
    row_i        = train.iloc[i]
    correct_text = str(row_i[row_i['answer']])
    _, I_i       = index.search(model.encode([str(row_i['prompt'])]), 5)
    retrieved    = [kb[idx] for idx in I_i[0]]
    if any(correct_text in doc for doc in retrieved):
        hits += 1
q7 = round((hits / 100) * 100, 1)
print(f" Q7: {q7}%")

# Q8 full RAG pipeline first 20 rows, MAP@3
def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

OPTS_LETTERS = ['A','B','C','D','E']
ap_scores = []
for i in range(20):
    row_i        = train.iloc[i]
    prompt_i     = str(row_i['prompt'])
    opts_i       = [str(row_i[o]) for o in OPTS_LETTERS]
    letter_map   = {str(row_i[o]): o for o in OPTS_LETTERS}
    _, I_i       = index.search(model.encode([prompt_i]), 5)
    docs_i       = [kb[idx] for idx in I_i[0]]
    ce_scores_i  = cross_encoder.predict([[prompt_i, doc] for doc in docs_i])
    best_doc     = docs_i[np.argmax(ce_scores_i)]
    rag_str_i    = f"Context: {best_doc} Question: {prompt_i}"
    res_i        = zs(rag_str_i, opts_i)
    ranked       = [letter_map[l] for l in res_i['labels']]
    ap_scores.append(apk(row_i['answer'], ranked))

q8 = round(sum(ap_scores) / len(ap_scores), 3)
print(f" Q8: {q8}")

wandb.log({
    "q1": q1, "q2": q2, "q3": q3, "q4": q4,
    "q5": q5, "q6": q6, "q7": q7, "q8": q8
})
wandb.finish()

 Q1: 0.384
retrieved order: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
 Q2: 10


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

cross encoder ranked order: [150, 1906, 847, 1693, 1269, 1532, 168, 576, 1701, 663]
 Q3: 1
 Q4: 216
 Q5: 0.989
 Q6: 0.529
 Q7: 73.0%
 Q8: 0.975
